## 1. Objective

The objective of this notebook is to construct the operator-level activity matrix that will be used as input for the first unsupervised clustering stage.

The matrix aggregates the cleaned North Region dataset by `ID_PLANO`, with one observation per operator. As of `data_pipeline/08_filter_medicina_de_grupo.ipynb`, that dataset is the Medicina de Grupo subset of the treated base (`NM_MODALIDADE_Medicina De Grupo == 1`), adopted as the project's official modeling population; see the filter notebook and Section 4.3.2.8 of the documentation for the decision. The selected features are intended to capture operator activity through volume, temporal presence, geographic presence, and operational diversity.

The resulting matrix will be used to identify active and marginal operators before constructing the final analytical dataset.

> **Data setup.** This notebook reads local files from `../data/`, which are not versioned (the repository's `.gitignore` blocks data files). No database connection is needed. The inputs are produced by the earlier notebooks from the public ANS open data (see the README, section *Running the Project*). Run `data_pipeline/08_filter_medicina_de_grupo.ipynb` first (or `python scripts/run_pipeline.py` from the repository root). On Google Colab, clone the repository first, install `requirements.txt`, and run the pipeline notebooks so that `data/` is populated in the Colab session.

## 2. Data Loading and Initial Validation

The analysis starts from the Medicina de Grupo subset of the cleaned item-level North Region dataset, produced by `data_pipeline/08_filter_medicina_de_grupo.ipynb`. Data cleaning, encoding, duplicate removal, and previously defined missing-value decisions were performed in the preceding data preparation stage; the modality filter is applied after that, and before any notebook in this file's chain.

In [1]:
from pathlib import Path

import pandas as pd

INPUT_PATH = Path("../data/processed/north_region_mama_2025_data_en_medicina_de_grupo.parquet")
if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        "Missing Medicina de Grupo base. Run data_pipeline/08_filter_medicina_de_grupo.ipynb "
        f"first. Expected path: {INPUT_PATH}"
    )

df_norte = pd.read_parquet(INPUT_PATH)

## 3. Definition of Activity Metrics

The activity matrix was designed to capture four complementary dimensions of operator activity:

| Dimension | Metric | Rationale |
|---|---|---|
| Volume | Total events | Measures the observed scale of healthcare activity |
| Temporal presence | Active months | Captures persistence of activity over time |
| Geographic presence | Beneficiary municipalities | Measures the geographic reach of the operator's beneficiary population |
| Geographic presence | Provider municipalities | Measures the geographic reach of service provision |
| Operational diversity | Distinct procedures | Captures the breadth of procedures associated with the operator |

In [2]:


# 3. Basic validation
print("Dataset shape:", df_norte.shape)
print("Unique operators:", df_norte["ID_PLANO"].nunique())

# 4. Build activity matrix
activity_matrix = (
    df_norte
    .groupby("ID_PLANO")
    .agg(
        total_eventos=("ID_EVENTO_ATENCAO_SAUDE", "nunique"),
        total_itens=("ID_EVENTO_ATENCAO_SAUDE", "size"),
        meses_ativos=("MES", "nunique"),
        municipios_beneficiarios=("CD_MUNICIPIO_BENEFICIARIO", "nunique"),
        municipios_prestadores=("CD_MUNICIPIO_PRESTADOR", "nunique"),
        procedimentos_distintos=("CD_PROCEDIMENTO", "nunique"),
    )
    .reset_index()
)

# 5. Explore the result
activity_matrix.describe()

Dataset shape: (70004, 119)
Unique operators: 1052


,total_eventos,total_itens,meses_ativos,municipios_beneficiarios,municipios_prestadores,procedimentos_distintos
count,1052.000000,1052.000000,1052.000000,1052.000000,1052.000000,1052.000000
mean,64.773764,66.543726,5.845057,4.083650,1.809886,2.915399
std,343.139147,344.830146,4.033370,7.813394,1.947937,2.498995
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,3.000000,3.000000,2.000000,1.000000,1.000000,1.000000
50%,7.000000,7.000000,5.000000,2.000000,1.000000,2.000000
75%,24.000000,25.000000,10.000000,4.000000,2.000000,4.000000
max,6859.000000,6859.000000,12.000000,112.000000,33.000000,18.000000


Each row in the resulting matrix represents one `ID_PLANO`. The aggregation transforms the original item-level data into an operator-level representation suitable for clustering.

In [3]:
activity_matrix["itens_por_evento"] = (
    activity_matrix["total_itens"] /
    activity_matrix["total_eventos"]
)

activity_matrix["itens_por_evento"].describe()

count    1052.000000
mean        1.054113
std         0.162574
min         1.000000
25%         1.000000
50%         1.000000
75%         1.005085
max         2.000000
Name: itens_por_evento, dtype: float64

In [4]:
activity_matrix["eventos_por_mes_ativo"] = (
    activity_matrix["total_eventos"] /
    activity_matrix["meses_ativos"]
)

Two derived metrics were initially created for exploratory analysis:

- `itens_por_evento` measures the average number of item records associated with each event.
- `eventos_por_mes_ativo` measures the average number of events during months in which the operator was active.

In [5]:
activity_matrix.describe()

,total_eventos,total_itens,meses_ativos,municipios_beneficiarios,municipios_prestadores,procedimentos_distintos,itens_por_evento,eventos_por_mes_ativo
count,1052.000000,1052.000000,1052.000000,1052.000000,1052.000000,1052.000000,1052.000000,1052.000000
mean,64.773764,66.543726,5.845057,4.083650,1.809886,2.915399,1.054113,6.320679
std,343.139147,344.830146,4.033370,7.813394,1.947937,2.498995,0.162574,28.772370
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,3.000000,3.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000
50%,7.000000,7.000000,5.000000,2.000000,1.000000,2.000000,1.000000,1.500000
75%,24.000000,25.000000,10.000000,4.000000,2.000000,4.000000,1.005085,2.717532
max,6859.000000,6859.000000,12.000000,112.000000,33.000000,18.000000,2.000000,571.583333


The descriptive statistics reveal a strongly right-skewed distribution for event volume and geographic/operational diversity. The median number of events per operator is substantially lower than the mean, indicating that a relatively small number of operators account for a large share of observed activity.

In [6]:
activity_matrix[[
    "total_eventos",
    "total_itens",
    "itens_por_evento"
]].corr()

,total_eventos,total_itens,itens_por_evento
total_eventos,1.000000,0.999657,-0.031134
total_itens,0.999657,1.000000,-0.027669
itens_por_evento,-0.031134,-0.027669,1.000000


In [7]:
activity_matrix[[
    "total_eventos",
    "total_itens"
]].corr()

,total_eventos,total_itens
total_eventos,1.000000,0.999657
total_itens,0.999657,1.000000


In [8]:
activity_features = [
    "total_eventos",
    "meses_ativos",
    "municipios_beneficiarios",
    "municipios_prestadores",
    "procedimentos_distintos",
    "eventos_por_mes_ativo"
]

In [9]:
correlation_matrix = activity_matrix[activity_features].corr()

correlation_matrix

,total_eventos,meses_ativos,municipios_beneficiarios,municipios_prestadores,procedimentos_distintos,eventos_por_mes_ativo
total_eventos,1.000000,0.251557,0.651569,0.399553,0.366825,0.998172
meses_ativos,0.251557,1.000000,0.468898,0.421560,0.701871,0.240031
municipios_beneficiarios,0.651569,0.468898,1.000000,0.723466,0.615378,0.644535
municipios_prestadores,0.399553,0.421560,0.723466,1.000000,0.557271,0.389875
procedimentos_distintos,0.366825,0.701871,0.615378,0.557271,1.000000,0.360658
eventos_por_mes_ativo,0.998172,0.240031,0.644535,0.389875,0.360658,1.000000


The correlation analysis was performed to identify redundant activity measures before clustering. `total_eventos` and `eventos_por_mes_ativo` showed an extremely high correlation (r = 0.998), indicating that they provide nearly identical information in the current dataset.

Therefore, `eventos_por_mes_ativo` was not retained as a clustering feature.

In [10]:
activity_features = [
    "total_eventos",
    "meses_ativos",
    "municipios_beneficiarios",
    "municipios_prestadores",
    "procedimentos_distintos"
]

activity_matrix[activity_features].skew().sort_values(ascending=False)

total_eventos               13.891603
municipios_beneficiarios     6.699220
municipios_prestadores       6.555148
procedimentos_distintos      2.153911
meses_ativos                 0.328727
dtype: float64

In [11]:
import numpy as np

log_features = [
    "total_eventos",
    "municipios_beneficiarios",
    "procedimentos_distintos",
    "municipios_prestadores"
]

for col in log_features:
    activity_matrix[f"{col}_LOG"] = np.log1p(activity_matrix[col])

The activity variables exhibited substantial positive skewness, particularly `total_eventos` (13.89), `municipios_beneficiarios` (6.70), `municipios_prestadores` (6.56), and `procedimentos_distintos` (2.15).

Because K-Means is sensitive to extreme observations, a `log1p` transformation was applied to these variables to reduce the influence of the long right tails.

In [12]:
activity_matrix[
    [f"{col}_LOG" for col in log_features]
].skew()

total_eventos_LOG               1.111231
municipios_beneficiarios_LOG    1.593614
procedimentos_distintos_LOG     0.816624
municipios_prestadores_LOG      2.091778
dtype: float64

In [13]:
pd.DataFrame({
    "before": activity_matrix[log_features].skew(),
    "after": activity_matrix[
        [f"{col}_LOG" for col in log_features]
    ].skew().values
})

,before,after
total_eventos,13.891603,1.111231
municipios_beneficiarios,6.699220,1.593614
procedimentos_distintos,2.153911,0.816624
municipios_prestadores,6.555148,2.091778


In [14]:
activity_features_final = [
    "total_eventos_LOG",
    "meses_ativos",
    "municipios_beneficiarios_LOG",
    "municipios_prestadores_LOG",
    "procedimentos_distintos_LOG"
]

In [15]:
activity_matrix[activity_features_final].describe()

,total_eventos_LOG,meses_ativos,municipios_beneficiarios_LOG,municipios_prestadores_LOG,procedimentos_distintos_LOG
count,1052.000000,1052.000000,1052.000000,1052.000000,1052.000000
mean,2.418563,5.845057,1.251676,0.927057,1.218304
std,1.494731,4.033370,0.710350,0.395535,0.509179
min,0.693147,1.000000,0.693147,0.693147,0.693147
25%,1.386294,2.000000,0.693147,0.693147,0.693147
50%,2.079442,5.000000,1.098612,0.693147,1.098612
75%,3.218876,10.000000,1.609438,1.098612,1.609438
max,8.833463,12.000000,4.727388,3.526361,2.944439


After reducing the skewness, the selected features were standardized using z-score standardization so that variables with different numerical ranges contribute comparably to the Euclidean distance used by K-Means.

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_activity = scaler.fit_transform(
    activity_matrix[activity_features_final]
)

In [17]:
scaled_activity = pd.DataFrame(
    X_activity,
    columns=activity_features_final
)

scaled_activity.describe()

,total_eventos_LOG,meses_ativos,municipios_beneficiarios_LOG,municipios_prestadores_LOG,procedimentos_distintos_LOG
count,1.052000e+03,1.052000e+03,1.052000e+03,1052.000000,1.052000e+03
mean,7.429629e-17,8.949326e-17,7.429629e-17,0.000000,4.457778e-16
std,1.000476e+00,1.000476e+00,1.000476e+00,1.000476,1.000476e+00
min,-1.154881e+00,-1.201814e+00,-7.866472e-01,-0.591656,-1.031869e+00
25%,-6.909331e-01,-9.537647e-01,-7.866472e-01,-0.591656,-1.031869e+00
50%,-2.269856e-01,-2.096160e-01,-2.155792e-01,-0.591656,-2.351793e-01
75%,5.356775e-01,1.030632e+00,5.038815e-01,0.433937,7.685315e-01
max,4.293716e+00,1.526731e+00,4.895287e+00,6.574743,3.391648e+00


In [18]:
activity_features_final

['total_eventos_LOG',
 'meses_ativos',
 'municipios_beneficiarios_LOG',
 'municipios_prestadores_LOG',
 'procedimentos_distintos_LOG']

In [19]:
X_activity.shape

(1052, 5)

In [20]:
activity_matrix[["ID_PLANO"] + activity_features_final].head()

,ID_PLANO,total_eventos_LOG,meses_ativos,municipios_beneficiarios_LOG,municipios_prestadores_LOG,procedimentos_distintos_LOG
0,1019993,2.079442,4,1.386294,1.098612,1.609438
1,1169447,2.484907,6,1.098612,1.386294,1.386294
2,1169667,1.098612,2,0.693147,0.693147,1.098612
3,1169698,5.468060,12,1.945910,0.693147,1.945910
4,1169764,1.386294,2,1.098612,0.693147,1.098612


In [21]:
print("Number of operators:", activity_matrix["ID_PLANO"].nunique())
print("Duplicate IDs:", activity_matrix["ID_PLANO"].duplicated().sum())
print("Missing values:")
print(activity_matrix[activity_features_final].isna().sum())
print("Final matrix shape:", X_activity.shape)

Number of operators: 1052
Duplicate IDs: 0
Missing values:
total_eventos_LOG               0
meses_ativos                    0
municipios_beneficiarios_LOG    0
municipios_prestadores_LOG      0
procedimentos_distintos_LOG     0
dtype: int64
Final matrix shape: (1052, 5)


The final activity matrix contains one observation per operator and five standardized clustering features. No duplicate operator IDs or unexpected missing values were identified in the selected features.

A sample of operators was manually checked against the original item-level dataset to verify that event counts, active months, municipality counts, and distinct procedure counts were aggregated correctly.

In [22]:
activity_matrix_scaled = pd.DataFrame(
    X_activity,
    columns=activity_features_final,
    index=activity_matrix.index
)

activity_matrix_scaled.insert(
    0,
    "ID_PLANO",
    activity_matrix["ID_PLANO"].values
)

## Manual Aggregation Validation

A sample of operators is selected and their activity metrics are recalculated directly from the original item-level dataset. The recalculated values are compared with the corresponding values in the aggregated activity matrix to verify that the aggregation logic was applied correctly.

In [23]:
sample_ids = (
    activity_matrix["ID_PLANO"]
    .sample(5, random_state=42)
    .tolist()
)

sample_ids

['1483742', '236911', '210316', '241199', '228816']

In [24]:
manual_validation = (
    df_norte[df_norte["ID_PLANO"].isin(sample_ids)]
    .groupby("ID_PLANO")
    .agg(
        total_eventos=("ID_EVENTO_ATENCAO_SAUDE", "nunique"),
        total_itens=("ID_EVENTO_ATENCAO_SAUDE", "size"),
        meses_ativos=("MES", "nunique"),
        municipios_beneficiarios=("CD_MUNICIPIO_BENEFICIARIO", "nunique"),
        municipios_prestadores=("CD_MUNICIPIO_PRESTADOR", "nunique"),
        procedimentos_distintos=("CD_PROCEDIMENTO", "nunique"),
    )
    .reset_index()
)

manual_validation

,ID_PLANO,total_eventos,total_itens,meses_ativos,municipios_beneficiarios,municipios_prestadores,procedimentos_distintos
0,1483742,6,6,4,2,1,1
1,210316,44,44,8,1,1,2
2,228816,13,13,8,1,1,2
3,236911,1,1,1,1,1,1
4,241199,31,31,12,1,1,4


In [25]:
matrix_validation = activity_matrix[
    activity_matrix["ID_PLANO"].isin(sample_ids)
][
    [
        "ID_PLANO",
        "total_eventos",
        "total_itens",
        "meses_ativos",
        "municipios_beneficiarios",
        "municipios_prestadores",
        "procedimentos_distintos",
    ]
].copy()

matrix_validation

,ID_PLANO,total_eventos,total_itens,meses_ativos,municipios_beneficiarios,municipios_prestadores,procedimentos_distintos
96,1483742,6,6,4,2,1,1
319,210316,44,44,8,1,1,2
522,228816,13,13,8,1,1,2
682,236911,1,1,1,1,1,1
731,241199,31,31,12,1,1,4


In [26]:
comparison = (
    manual_validation
    .merge(
        matrix_validation,
        on="ID_PLANO",
        suffixes=("_manual", "_matrix")
    )
)

comparison

,ID_PLANO,total_eventos_manual,total_itens_manual,meses_ativos_manual,municipios_beneficiarios_manual,municipios_prestadores_manual,procedimentos_distintos_manual,total_eventos_matrix,total_itens_matrix,meses_ativos_matrix,municipios_beneficiarios_matrix,municipios_prestadores_matrix,procedimentos_distintos_matrix
0,1483742,6,6,4,2,1,1,6,6,4,2,1,1
1,210316,44,44,8,1,1,2,44,44,8,1,1,2
2,228816,13,13,8,1,1,2,13,13,8,1,1,2
3,236911,1,1,1,1,1,1,1,1,1,1,1,1
4,241199,31,31,12,1,1,4,31,31,12,1,1,4


In [27]:
assert (
    comparison["total_eventos_manual"]
    == comparison["total_eventos_matrix"]
).all()

assert (
    comparison["total_itens_manual"]
    == comparison["total_itens_matrix"]
).all()

assert (
    comparison["meses_ativos_manual"]
    == comparison["meses_ativos_matrix"]
).all()

assert (
    comparison["municipios_beneficiarios_manual"]
    == comparison["municipios_beneficiarios_matrix"]
).all()

assert (
    comparison["municipios_prestadores_manual"]
    == comparison["municipios_prestadores_matrix"]
).all()

assert (
    comparison["procedimentos_distintos_manual"]
    == comparison["procedimentos_distintos_matrix"]
).all()

print("Manual validation passed for all sampled operators.")

Manual validation passed for all sampled operators.


The aggregation was validated using a reproducible manual check. Five operators were randomly sampled, their activity metrics were independently recalculated from the original item-level dataset, and the results were compared against the aggregated activity matrix. All sampled operators matched across the evaluated metrics.

## Extreme Activity Patterns

The activity metrics show highly right-skewed distributions, with a small number of operators presenting substantially higher activity than the rest of the population. These observations were inspected to assess their potential influence on the first K-Means clustering.

In [28]:
top_volume = (
    activity_matrix
    .sort_values("total_eventos", ascending=False)
    [[
        "ID_PLANO",
        "total_eventos",
        "meses_ativos",
        "municipios_beneficiarios",
        "municipios_prestadores",
        "procedimentos_distintos"
    ]]
    .head(10)
)

top_volume

,ID_PLANO,total_eventos,meses_ativos,municipios_beneficiarios,municipios_prestadores,procedimentos_distintos
705,239258,6859,12,96,18,11
47,1173218,5792,12,19,1,10
399,216532,3753,12,58,8,9
363,214111,2461,12,38,4,11
795,248834,1617,12,56,5,8
717,239430,1576,12,57,15,7
712,239400,1431,12,39,10,8
1048,572290,1268,9,20,2,7
715,239426,1249,12,50,12,7
725,239496,1230,12,39,10,8


In [29]:
top_beneficiary_reach = (
    activity_matrix
    .sort_values("municipios_beneficiarios", ascending=False)
    [[
        "ID_PLANO",
        "total_eventos",
        "meses_ativos",
        "municipios_beneficiarios",
        "municipios_prestadores",
        "procedimentos_distintos"
    ]]
    .head(10)
)

top_beneficiary_reach

,ID_PLANO,total_eventos,meses_ativos,municipios_beneficiarios,municipios_prestadores,procedimentos_distintos
515,226458,520,12,112,5,9
705,239258,6859,12,96,18,11
944,418057,520,12,60,16,13
399,216532,3753,12,58,8,9
717,239430,1576,12,57,15,7
795,248834,1617,12,56,5,8
715,239426,1249,12,50,12,7
713,239402,1012,12,45,13,5
977,420883,505,12,43,10,16
174,163444,396,12,41,33,10


In [30]:
top_procedure_diversity = (
    activity_matrix
    .sort_values("procedimentos_distintos", ascending=False)
    [[
        "ID_PLANO",
        "total_eventos",
        "meses_ativos",
        "municipios_beneficiarios",
        "municipios_prestadores",
        "procedimentos_distintos"
    ]]
    .head(10)
)

top_procedure_diversity

,ID_PLANO,total_eventos,meses_ativos,municipios_beneficiarios,municipios_prestadores,procedimentos_distintos
562,230117,624,12,29,5,18
546,229022,138,12,8,2,16
977,420883,505,12,43,10,16
456,219776,320,12,19,5,15
968,420026,1027,12,31,6,14
444,219757,201,12,29,6,14
457,219777,184,12,19,4,13
944,418057,520,12,60,16,13
957,419337,79,12,17,7,13
989,422479,153,12,10,4,12


In [31]:
activity_matrix[
    [
        "total_eventos",
        "municipios_beneficiarios",
        "municipios_prestadores",
        "procedimentos_distintos"
    ]
].quantile([0.50, 0.90, 0.95, 0.99, 1.00])

,total_eventos,municipios_beneficiarios,municipios_prestadores,procedimentos_distintos
0.50,7.00,2.00,1.0,2.0
0.90,87.90,8.00,3.0,6.0
0.95,190.30,15.45,5.0,8.0
0.99,1099.51,39.00,9.0,12.0
1.00,6859.00,112.00,33.0,18.0


### Interpretation

The aggregated activity metrics exhibit strong right-skewness and a long upper tail. The median operator has 7 observed events, while the 99th percentile reaches approximately 1,100 events and the maximum reaches 6,859 events. Similar behavior is observed for geographic reach and procedural diversity, with substantial differences between the typical operator and the most active operators.

These extreme observations were not removed because high activity is directly relevant to the objective of distinguishing active operators from marginal ones. Instead, the influence of extreme values was reduced through `log1p` transformation before standardization and clustering.

In [32]:
raw_output_path = "../data/interim/north_region_mama_2025_activity_matrix.parquet"

activity_matrix.to_parquet(
    raw_output_path,
    index=False
)

In [33]:
scaled_output_path = "../data/processed/north_region_mama_2025_activity_matrix_scaled.parquet"

activity_matrix_scaled.to_parquet(
    scaled_output_path,
    index=False
)

## References

1. Agência Nacional de Saúde Suplementar (ANS). Dados abertos: TISS Ambulatorial [Internet]. Brasília: ANS; 2026 [cited 2026 Sep 11]. Available from: https://dadosabertos.ans.gov.br/FTP/PDA/TISS/AMBULATORIAL/
2. Pedregosa F, Varoquaux G, Gramfort A, Michel V, Thirion B, Grisel O, et al. Scikit-learn: machine learning in Python. J Mach Learn Res. 2011;12:2825-30.
3. James G, Witten D, Hastie T, Tibshirani R. An introduction to statistical learning. New York: Springer; 2013.
